In [5]:
## imports
import numpy as np
import sys
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
## Constants
from config import (
    SCREEN_WIDTH, SCREEN_HEIGHT, PRIMARY_COLOR, ACCENT_COLOR, TEXT_COLOR, GRAY, BLACK, BG_COLOR, WHITE
)

### Create a reusable buttons class

In [ ]:
class Button:

    ## button constructor
    def __init__(self, dimensions, text, callback, 
                 font, text_color=TEXT_COLOR, 
                 base_color=PRIMARY_COLOR, 
                 hover_color=ACCENT_COLOR):
        
        '''
        rect (tuple): (x, y, width, height) dimensions for the button
        text (str): text label displayed on the button
        callback (function): function to be executed when the button is clicked
        font (pygame.font.Font): Pygame Font object used for rendering text
        base_color (tuple, opt): base color of the button
        hover_color (tuple, opt): color of the button when the mouse hovers over it
        text_color (tuple, opt): color of the text
        '''

        self.rect = pygame.rect(dimensions)
        self.text = text
        self.callback = callback
        self.font = font
        self.base_color = base_color
        self.hover_color = hover_color
        self.current_color = base_color
        self.text_color = text_color
        self.is_hovered = False

    ## draw the button so it appears on the screen
    def draw(self, surface):

        '''
        params:
            surface (pygame.Surface): Pygame surface on which button will be drawn
        '''

        self.current_color = self.hover_color if self.is_hovered else self.base_color
        
        # draw button with rounded corners
        pygame.draw.rect(surface=surface,
                         color=self.current_color,
                         rect=self.rect,
                         border_radius=4)
    
        # draw text of the button in the center of the "rectangle"
        text_surface = self.font.render(text=self.text,
                                        antialias=True,
                                        color=self.text_color)
        text_shape = text_surface.get_rect(center=self.rect.center)
        surface.blit(source=text_surface, dest=text_shape)

    # handle all events for the button (so far just hover or click)
    def handle_event(self, event):
        
        '''
        params:
            event (pygame.event.Event): Pygame event (e.g., MOUSEMOTION, MOUSEBUTTONDOWN).
        returns:
            any: return value of the callback function, or None.
        '''
        
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1 and self.rect.collidepoint(event.pos):
                return self.callback() # defined callback when the button is clicked
            
        if event.type == pygame.MOUSEMOTION:
            self.is_hovered = self.rect.collidepoint(event.pos)

        return None

### Creating a Reusable Menu Class

In [ ]:
class Menu:
    def __init__(self, menu_manager, title, text, user_options, user_closable=False):

        '''
            manager (MenuManager): MenuManager instance
            title_text (str): main title displayed at the top of the menu
            body_text (str):ain message inside the menu
            options (list): list of tuples => (button_text (str), callback (function))
            user_closable (bool): have a clickable 'x' button so users can close the menu if they are allowed
        '''

        self.menu_manager = menu_manager
        self.title = title
        self.text = text
        self.user_options = user_options
        self.user_closable=user_closable
        self.buttons = []
        self.close_button = None
        
        self.title_font = pygame.font.Font(None, 40)
        self.text_font = pygame.font.Font(None, 24)
        self.button_text_font = pygame.font.Font(None, 30)

        # define the geometry for the menu
        self.width = 500
        self.height = 400
        self.x = (SCREEN_WIDTH - self.width) // 2 # so that the menu is centered
        self.y = (SCREEN_HEIGHT - self.height) // 2
        self.rect = pygame.Rect(self.x, self.y, self.width, self.height)

        self._menu_setup() # call the protected helper function to organize the menu interface

    # create wrapper so that after the callback from the dev is executed, we close out the memu
    def _create_button_callback(self, custom_callback):

        def wrapped_callback():
            result = custom_callback()
            self.manager.close_menu()
            return result

        return wrapped_callback
    
    def _menu_setup(self):

        # DONT HARD CODE FIXME
        button_width = 200
        button_height = 40
        spacing = 20

        # calculate how to center the buttons
        starting_x_pos = self.x + (self.width - (button_width * len(self.user_options)) + (spacing * (len(self.user_options) - 1))) // 2

        button_y = self.y + self.height - 70 # DONT HARD CODE FIXME

        # create the buttons for all the ones listed for the menu
        for i, (button_text, callback) in enumerate(self.user_options):

            button_x = starting_x_pos + (i * (button_width + spacing))
            button_box = (button_x, button_y, button_width, button_height)
            
            wrapped_callback = self._create_button_callback(callback)

            button = Button(dimensions=button_box, 
                            text=button_text, 
                            callback=wrapped_callback, 
                            font=self.button_text_font)
            self.buttons.append(button)

        # The 'x' button to close menu
        if self.user_closable:
            ## DONT HARD CODE THE VALS FIXME
            close_size = 20
            close_x = self.x + self.width - close_size - 10
            close_y = self.y + 10
            close_dims = (close_x, close_y, close_size, close_size)
            
            self.close_button = Button(dimensions=close_dims, 
                                       text="x", 
                                       callback=self.manager.close_menu, 
                                       font=self.button_text_font,
                                       color=BG_COLOR, 
                                       hover_color=GRAY, 
                                       text_color=BLACK)
    
    # so that the text wraps around and is not cut off if too long
    def _wrap_text(self, surface, text, font, x, y, max_width):
        
        '''
        params:
            surface (pygame.Surface): surface to draw on
            text (str): string  to wrap and display
            font (pygame.font.Font): font object to use for rendering
            x (int): starting x-coordinate for the text
            y (int): starting y-coordinate for the text
            max_width (int): max width of the area before wrapping to the next line
        '''

        space = ' '
        words = text.split(space)
        lines = [] # curating line by line to fit the button width
        current_line = []
        for word in words:
            test = space.join(current_line + [word])

            # if adding a word is within the maximum width, add it
            if font.size(test)[0] < max_width:
                current_line.append(word)
            # if not, add a space and move to a new line
            else: 
                lines.append(space.join(current_line))
                current_line = [word]
        lines.append(space.join(current_line))

        for line in lines:
            text_surface = font.render(text=line, 
                                       antialias=True, 
                                       color=TEXT_COLOR)
            surface.blit(source=text_surface, 
                         dest=(x, y))
            
            y += font.get_linesize()

    # so that the whole menu container shows up on the screen
    def draw(self, surface):
        
        '''
        params:
            surface (pygame.Surface): The Pygame surface (e.g., the main screen) to draw the menu onto.
        '''
        
        # dim the main window
        overlay = pygame.Surface((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.SRCALPHA)
        overlay.fill((0, 0, 0, 150)) ## DONT HARD CODE FIXME
        surface.blit(overlay, (0, 0))

        # draw the main menu window that pops up
        pygame.draw.rect(surface=surface, 
                         color=WHITE, 
                         rect=self.rect, 
                         border_radius=6)
        
        pygame.draw.rect(surface=surface, 
                         color=PRIMARY_COLOR, 
                         recr=self.rect, 
                         width=3, 
                         border_radius=6)

        # draw the title text
        title_surface = self.title_font.render(self.title, True, TEXT_COLOR)
        title_shape = title_surface.get_rect(centerx=self.rect.centerx, y=self.y + 30) # 30 is a placeholder FIXME
        surface.blit(title_surface, title_shape)
        
        # draw the main message text so that it doesn't overflow if the button is to
        self._wrap_text(surface=surface, 
                        text=self.text, 
                        font=self.text_font, 
                        x=self.x + 30, 
                        y=self.y + 90, 
                        max_width=self.width - 60) ## 30 pixel margin on both side dont hard code FIXME

        # draw the buttons
        for button in self.buttons:
            button.draw(surface=surface)
            
        # draw the close button
        if self.close_button:
            self.close_button.draw(surface=surface)

    # handle events for all the buttons possible
    def handle_event(self, event):
        
        '''
        params:
            event (pygame.event.Event): Pygame event to respond to
        '''

        for button in self.buttons:
            button.handle_event(event) # pass in the button's event handler that we created
            
        if self.close_button:
            self.close_button.handle_event(event)

### MenuManager class to clean up and handle transitioning between menu display and regular main screen

In [ ]:
class MenuManager:

    def __init__(self, main_data):
        
        '''
        params:
            main_data (list): list of tuples => (button_text (str), callback_function (function)) for main screen
        '''

        self.main_buttons = []
        self.active_menu = None
        self.main_font = pygame.font.Font(None, 36)
        
        self._main_setup(main_data)

    def _main_setup(self, data):
        
        '''
        Args:
            data (list): list of (text, callback) tuples to create the corresponding buttons
        '''

        ## values hard coded for now FIXME
        button_width = 250
        button_height = 60
        start_y = 150
        spacing = 20

        for i, (text, callback) in enumerate(data):
            rect = ((SCREEN_WIDTH - button_width) // 2, 
                    start_y + (i * (button_height + spacing)), 
                    button_width, 
                    button_height)
            
            button = Button(dimensions=rect, 
                            text=text, 
                            callback=callback, 
                            font=self.main_font, 
                            color=PRIMARY_COLOR, 
                            hover_color=ACCENT_COLOR)
            
            self.main_buttons.append(button)

    def open_menu(self, menu):

        '''
        Args:
            menu (Menu): menu object instance to activate and draw
        '''

        self.active_menu = menu

    def close_menu(self):

        self.active_menu = None
        print("menu closed")

    def draw(self, surface):

        '''
        Args:
            surface (pygame.Surface): The Pygame surface (e.g., the main screen) to draw onto.
        '''
        
        # draw main menu buttons
        for button in self.main_buttons:
            button.draw(surface)
            
        # draw modal menu last, so it appears on top
        if self.active_menu:
            self.active_menu.draw(surface)

    def handle_event(self, event):

        '''
        params:
            event (pygame.event.Event): Pygame event to respond to
        '''

        # Events are only handled by the menu if it is active
        if self.active_menu:
            self.active_menu.handle_event(event)
        else:
            # If no menu is active, allow interaction with main buttons.
            for button in self.main_buttons:
                button.handle_event(event)